# Data Exploration

## Read

In [151]:
import pandas as pd
abm_path = './data/abm.csv'
card_path = './data/card.csv'
cheque_path = './data/cheque.csv'
eft_path = './data/eft.csv'
emt_path = './data/emt.csv'
western_union_path = './data/westernunion.csv'
wire_path = './data/wire.csv'
kyc_individuals_path = './data/kyc_individual.csv'
kyc_small_business_path = './data/kyc_smallbusiness.csv'

abm = pd.read_csv(abm_path)
card_df = pd.read_csv(card_path)
cheque_df = pd.read_csv(cheque_path)
eft_df = pd.read_csv(eft_path)
emt_df = pd.read_csv(emt_path)
western_union_df = pd.read_csv(western_union_path)
wire_df = pd.read_csv(wire_path)
kyc_individuals_df = pd.read_csv(kyc_individuals_path)
kyc_small_business_df = pd.read_csv(kyc_small_business_path)

## View Transactions that have 'recieved' meta data

In [152]:
cheque_df.head()

,transaction_id,customer_id,amount_cad,debit_credit,transaction_datetime
0,CHQ241101366206,SYNID0200605005,813.90,C,2024-11-01
1,CHQ241101444612,SYNID0200217537,2851.12,C,2024-11-01
2,CHQ241101694712,SYNID0100054434,1602.20,D,2024-11-01
3,CHQ241101203093,SYNID0101266181,108.95,D,2024-11-01
4,CHQ241101799936,SYNID0200927212,4993.85,D,2024-11-01


In [153]:
wire_df.head()

,transaction_id,customer_id,amount_cad,debit_credit,transaction_datetime
0,2411018567515752,SYNID0106350653,3070.13,C,2024-11-01
1,2411011617097052,SYNID0104386115,630.08,C,2024-11-01
2,2411017551414972,SYNID0200341758,21353.70,C,2024-11-01
3,2411011591775144,SYNID0107845453,1015.64,C,2024-11-01
4,2411015685130952,SYNID0108075156,128943.27,C,2024-11-01


In [154]:
western_union_df.head()

,transaction_id,customer_id,amount_cad,debit_credit,transaction_datetime
0,WU70151224,SYNID0101092403,228.71,D,2024-11-01 04:04:47
1,WU65427055,SYNID0106130686,127.78,D,2024-11-01 06:34:20
2,WU72536621,SYNID0101383552,123.21,D,2024-11-01 06:34:58
3,WU17761185,SYNID0107295306,977.30,D,2024-11-01 08:12:28
4,WU63304758,SYNID0105925328,968.66,D,2024-11-01 08:14:23


In [155]:
cheque_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240548 entries, 0 to 240547
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   transaction_id        240548 non-null  object 
 1   customer_id           240548 non-null  object 
 2   amount_cad            240548 non-null  float64
 3   debit_credit          240548 non-null  object 
 4   transaction_datetime  240548 non-null  object 
dtypes: float64(1), object(4)
memory usage: 9.2+ MB


In [156]:
for df in [cheque_df, eft_df, emt_df, western_union_df, wire_df, card_df]:
    assert df['transaction_id'].nunique() == len(df)
# Runs no problem, doesn't seem to have interconnected transactions

In [157]:
mask = (cheque_df['transaction_id'] == 'CHQ241101366206' )
cheque_df.loc[mask, 'customer_id'] 

0    SYNID0200605005
Name: customer_id, dtype: object

In [158]:
import pandas as pd

def find_interconnected_transactions(df, label="df"):
    credits = df[df['debit_credit'] == 'C'].copy()
    debits = df[df['debit_credit'] == 'D'].copy()

    matches = pd.merge(
        credits, 
        debits, 
        on=['amount_cad', 'transaction_datetime'], 
        suffixes=('_recip', '_sender')
    )
    
    internal_transfers = matches[matches['customer_id_recip'] != matches['customer_id_sender']]
    
    print(f"--- Analysis for {label} ---")
    print(f"Total Transactions: {len(df)}")
    print(f"Interconnected Pairs Found: {len(internal_transfers)}")
    
    return internal_transfers

internal_network_results = {}

transaction_dfs = {
    "cheques": cheque_df,
    "eft": eft_df,
    "emt": emt_df,
    "western_union": western_union_df,
    "wire": wire_df,
    "card": card_df
}

for name, df in transaction_dfs.items():
    internal_network_results[name] = find_interconnected_transactions(df, name)

internal_network_results['cheques']

--- Analysis for cheques ---
Total Transactions: 240548
Interconnected Pairs Found: 2199
--- Analysis for eft ---
Total Transactions: 1070698
Interconnected Pairs Found: 7743
--- Analysis for emt ---
Total Transactions: 845996
Interconnected Pairs Found: 4
--- Analysis for western_union ---
Total Transactions: 2142
Interconnected Pairs Found: 0
--- Analysis for wire ---
Total Transactions: 4956
Interconnected Pairs Found: 0
--- Analysis for card ---
Total Transactions: 3553303
Interconnected Pairs Found: 3


,transaction_id_recip,customer_id_recip,amount_cad,debit_credit_recip,transaction_datetime,transaction_id_sender,customer_id_sender,debit_credit_sender
0,CHQ241101991734,SYNID0200015187,537.00,C,2024-11-01,CHQ241101994724,SYNID0200200283,D
1,CHQ241101883896,SYNID0103580481,54.24,C,2024-11-01,CHQ241101009998,SYNID0102341588,D
2,CHQ241101883896,SYNID0103580481,54.24,C,2024-11-01,CHQ241101581829,SYNID0101117932,D
3,CHQ241101874471,SYNID0200713077,168.22,C,2024-11-01,CHQ241101638237,SYNID0105311354,D
4,CHQ241101872166,SYNID0200509059,1199.58,C,2024-11-01,CHQ241101301247,SYNID0105812401,D
...,...,...,...,...,...,...,...,...
2194,CHQ250131221129,SYNID0100837292,103.43,C,2025-01-31,CHQ250131863238,SYNID0200482923,D
2195,CHQ250131354890,SYNID0200406497,394.96,C,2025-01-31,CHQ250131277717,SYNID0102769579,D
2196,CHQ250131489351,SYNID0200426304,1967.03,C,2025-01-31,CHQ250131324534,SYNID0200820059,D
2197,CHQ250131272376,SYNID0109519568,39.13,C,2025-01-31,CHQ250131119081,SYNID0200870329,D


In [159]:
all_network_customers = set()
print(f"{'Channel':<20} | {'Unique Customers':<15}")
print("-" * 40)
for channel, df in internal_network_results.items():
    if not df.empty:
        senders = set(df['customer_id_sender'])
        recipients = set(df['customer_id_recip'])
        channel_customers = senders.union(recipients)
        all_network_customers.update(channel_customers)
        
        print(f"{channel:<20} | {len(channel_customers):<15}")
    else:
        print(f"{channel:<20} | 0")
print("-" * 40)
print(f"Total Unique Customers in Network: {len(all_network_customers)}")

Channel              | Unique Customers
----------------------------------------
cheques              | 3301           
eft                  | 10016          
emt                  | 8              
western_union        | 0
wire                 | 0
card                 | 6              
----------------------------------------
Total Unique Customers in Network: 12724


## A graph based approach to transaction monitoring seems not feasible. As we see the C or D, but we don't know which accounts they went to. 
* Reminder to reach out to scotia email asking for clarification on this 

In [160]:
labels_df = pd.read_csv('data/labels.csv')

In [161]:
labeled_customers = set(labels_df['customer_id'])

In [162]:
lst = ['cheque_df', 'eft_df', 'emt_df', 'western_union_df', 'wire_df', 'card_df']
total_labeled_transaction = 0
for i, transaction_type in enumerate([cheque_df, eft_df, emt_df, western_union_df, wire_df, card_df]):
    count_transaction_type = len(
    transaction_type.loc[transaction_type['customer_id'].isin(labeled_customers), 'transaction_id'].unique()
    )
    total_labeled_transaction += count_transaction_type
    print(f'Total Labeled {lst[i]} Transactions:', count_transaction_type)
print('Total Labeled Transactions:', total_labeled_transaction)

Total Labeled cheque_df Transactions: 2850
Total Labeled eft_df Transactions: 13006
Total Labeled emt_df Transactions: 9173
Total Labeled western_union_df Transactions: 19
Total Labeled wire_df Transactions: 52
Total Labeled card_df Transactions: 31636
Total Labeled Transactions: 56736


In [163]:
kyc_individuals_df.head(n=len(kyc_individuals_df))

,customer_id,country,province,city,gender,marital_status,occupation_code,income,birth_date,onboard_date
0,SYNID0100000167,CA,ON,TORONTO,NaN,Married,10019,48886.0,1972-01-30,2011-09-20
1,SYNID0100000431,CA,NaN,other,FEMALE,Married,72310,NaN,1988-03-20,2018-06-04
2,SYNID0100000485,CA,ON,BRAMPTON,FEMALE,Widowed,RETIRED,19998.0,1935-05-02,1997-07-12
3,SYNID0100000539,CA,NaN,other,MALE,Widowed,RETIRED,39417.0,1944-11-04,1985-05-14
4,SYNID0100000932,CA,ON,TORONTO,FEMALE,Married,RETIRED,34182.0,1963-09-14,2012-07-09
...,...,...,...,...,...,...,...,...,...,...
53094,SYNID0109999056,CA,ON,GUELPH,MALE,NaN,OTHER,NaN,1999-11-27,2020-07-26
53095,SYNID0109999204,CA,QC,MONTREAL,FEMALE,NaN,STUDENT,NaN,2006-04-08,2024-09-08
53096,SYNID0109999322,CA,ON,NIAGARA FALLS,FEMALE,Married,52120,NaN,1995-12-06,2024-11-25
53097,SYNID0109999736,CA,ON,ST CATHARINES,MALE,Single,14102,NaN,2007-02-19,2017-07-03


In [164]:
kyc_small_business_df.head(n=len(kyc_small_business_df))

,customer_id,country,province,city,industry_code,employee_count,sales,established_date,onboard_date
0,SYNID0200000024,CA,ON,BRAMPTON,9699,5.0,181876.0,2022-12-09,2022-12-26
1,SYNID0200000050,CA,ON,TORONTO,4214,1.0,250009.0,1991-02-24,1996-06-17
2,SYNID0200000104,CA,AB,EDMONTON,7215,2.0,217904.0,2011-06-20,NaN
3,SYNID0200000167,CA,NaN,other,8649,1.0,0.0,2024-02-20,2024-11-27
4,SYNID0200000345,CA,NaN,other,9861,NaN,NaN,2003-12-08,2003-12-18
...,...,...,...,...,...,...,...,...,...
8306,SYNID0200999416,CA,NaN,other,8641,0.0,NaN,1994-04-01,NaN
8307,SYNID0200999425,CA,ON,TORONTO,4569,1.0,0.0,2022-08-22,2022-11-06
8308,SYNID0200999482,CA,NaN,other,4279,1.0,0.0,2008-07-06,2008-02-29
8309,SYNID0200999834,CA,AB,EDMONTON,4599,1.0,29867.0,2023-02-20,2023-10-21


In [165]:
lst = ['cheque_df', 'eft_df', 'emt_df', 'western_union_df', 'wire_df', 'card_df']
total_labeled_transaction = 0
for i, transaction_type in enumerate([cheque_df, eft_df, emt_df, western_union_df, wire_df, card_df]):
    print(lst[i], transaction_type.columns)

cheque_df Index(['transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
       'transaction_datetime'],
      dtype='object')
eft_df Index(['transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
       'transaction_datetime'],
      dtype='object')
emt_df Index(['transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
       'transaction_datetime'],
      dtype='object')
western_union_df Index(['transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
       'transaction_datetime'],
      dtype='object')
wire_df Index(['transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
       'transaction_datetime'],
      dtype='object')
card_df Index(['transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
       'transaction_datetime', 'merchant_category', 'ecommerce_ind', 'country',
       'province', 'city'],
      dtype='object')


In [166]:
print(kyc_individuals_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53099 entries, 0 to 53098
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      53099 non-null  object 
 1   country          53099 non-null  object 
 2   province         35664 non-null  object 
 3   city             53099 non-null  object 
 4   gender           47693 non-null  object 
 5   marital_status   47952 non-null  object 
 6   occupation_code  53012 non-null  object 
 7   income           38421 non-null  float64
 8   birth_date       49573 non-null  object 
 9   onboard_date     49074 non-null  object 
dtypes: float64(1), object(9)
memory usage: 4.1+ MB
None


In [167]:
kyc_individuals_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53099 entries, 0 to 53098
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      53099 non-null  object 
 1   country          53099 non-null  object 
 2   province         35664 non-null  object 
 3   city             53099 non-null  object 
 4   gender           47693 non-null  object 
 5   marital_status   47952 non-null  object 
 6   occupation_code  53012 non-null  object 
 7   income           38421 non-null  float64
 8   birth_date       49573 non-null  object 
 9   onboard_date     49074 non-null  object 
dtypes: float64(1), object(9)
memory usage: 4.1+ MB


In [168]:
kyc_small_business_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8311 entries, 0 to 8310
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       8311 non-null   object 
 1   country           8311 non-null   object 
 2   province          5429 non-null   object 
 3   city              8311 non-null   object 
 4   industry_code     8291 non-null   object 
 5   employee_count    7510 non-null   float64
 6   sales             7459 non-null   float64
 7   established_date  7901 non-null   object 
 8   onboard_date      7557 non-null   object 
dtypes: float64(2), object(7)
memory usage: 584.5+ KB


In [169]:
lst = ['Cheques Transactions', 'Electronic Funds Transfer Transactions', 'Email Money Transfer Transactions', 'Western Union Transactions', 'Wire Transactions', 'Card Transactions']
total_labeled_transaction = 0
for i, transaction_type in enumerate([cheque_df, eft_df, emt_df, western_union_df, wire_df, card_df]):
    print(f'**{lst[i]}**')
    print(transaction_type.info())

**Cheques Transactions**
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240548 entries, 0 to 240547
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   transaction_id        240548 non-null  object 
 1   customer_id           240548 non-null  object 
 2   amount_cad            240548 non-null  float64
 3   debit_credit          240548 non-null  object 
 4   transaction_datetime  240548 non-null  object 
dtypes: float64(1), object(4)
memory usage: 9.2+ MB
None
**Electronic Funds Transfer Transactions**
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1070698 entries, 0 to 1070697
Data columns (total 5 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   transaction_id        1070698 non-null  object 
 1   customer_id           1070698 non-null  object 
 2   amount_cad            1070698 non-null  float64
 3   debit_credit       

# Data Wrangling

In [170]:
cheque_df['transaction_type'] = 'cheque'
eft_df['transaction_type'] = 'eft'
emt_df['transaction_type'] = 'emt'
western_union_df['transaction_type'] = 'western_union'
wire_df['transaction_type'] = 'wire'
card_df['transaction_type'] = 'card'

all_transactions = pd.concat([cheque_df, eft_df, emt_df, 
                              western_union_df, wire_df, card_df])

In [171]:
all_transactions['transaction_datetime'] = pd.to_datetime(
    all_transactions['transaction_datetime'],
    format='mixed',
    errors='coerce'
)

customer_features = all_transactions.groupby('customer_id').agg({
    'transaction_id': 'count',
    'amount_cad': ['sum', 'mean', 'median', 'std', 'min', 'max'],
    'transaction_datetime': ['min', 'max']  # Just min/max, no lambda
})

# Flatten column names
customer_features.columns = ['_'.join(col).strip() for col in customer_features.columns.values]
customer_features.reset_index(inplace=True)

# Rename for clarity
customer_features.rename(columns={
    'transaction_id_count': 'total_transaction_count',
    'amount_cad_sum': 'total_volume_cad',
    'amount_cad_mean': 'avg_transaction_amount',
    'amount_cad_median': 'median_transaction_amount',
    'amount_cad_std': 'std_transaction_amount',
    'amount_cad_min': 'min_transaction_amount',
    'amount_cad_max': 'max_transaction_amount',
    'transaction_datetime_min': 'first_transaction_date',
    'transaction_datetime_max': 'last_transaction_date'
}, inplace=True)

unique_days = all_transactions.groupby('customer_id')['transaction_datetime'].apply(
    lambda x: x.dt.date.nunique()
).reset_index()
unique_days.columns = ['customer_id', 'unique_active_days']

customer_features = customer_features.merge(unique_days, on='customer_id')

customer_features['account_activity_days'] = (
    customer_features['last_transaction_date'] - 
    customer_features['first_transaction_date']
).dt.days + 1

customer_features['activity_density'] = (
    customer_features['unique_active_days'] / 
    customer_features['account_activity_days']
)

customer_features['avg_transactions_per_day'] = (
    customer_features['total_transaction_count'] / 
    customer_features['account_activity_days']
)

transaction_type_breakdown = all_transactions.groupby(
    ['customer_id', 'transaction_type']
).size().unstack(fill_value=0)

transaction_type_breakdown.columns = [f'{col}_count' for col in transaction_type_breakdown.columns]
transaction_type_breakdown.reset_index(inplace=True)

customer_features = customer_features.merge(transaction_type_breakdown, on='customer_id', how='left')

debit_credit_breakdown = all_transactions.groupby(
    ['customer_id', 'debit_credit']
).agg({'amount_cad': ['count', 'sum']}).unstack(fill_value=0)

# Flatten column names
debit_credit_breakdown.columns = ['_'.join(col).strip() for col in debit_credit_breakdown.columns.values]
debit_credit_breakdown.reset_index(inplace=True)

# Rename for clarity
rename_map = {}
for col in debit_credit_breakdown.columns:
    if 'credit' in col and 'count' in col:
        rename_map[col] = 'credit_count'
    elif 'credit' in col and 'sum' in col:
        rename_map[col] = 'credit_volume'
    elif 'debit' in col and 'count' in col:
        rename_map[col] = 'debit_count'
    elif 'debit' in col and 'sum' in col:
        rename_map[col] = 'debit_volume'

debit_credit_breakdown.rename(columns=rename_map, inplace=True)

customer_features = customer_features.merge(debit_credit_breakdown, on='customer_id', how='left')

customer_features.fillna(0, inplace=True)

print(f"Customer features shape: {customer_features.shape}")
print(f"Columns: {list(customer_features.columns)}")
print(customer_features.head())

Customer features shape: (61403, 24)
Columns: ['customer_id', 'total_transaction_count', 'total_volume_cad', 'avg_transaction_amount', 'median_transaction_amount', 'std_transaction_amount', 'min_transaction_amount', 'max_transaction_amount', 'first_transaction_date', 'last_transaction_date', 'unique_active_days', 'account_activity_days', 'activity_density', 'avg_transactions_per_day', 'card_count', 'cheque_count', 'eft_count', 'emt_count', 'western_union_count', 'wire_count', 'amount_cad_count_C', 'amount_cad_count_D', 'amount_cad_sum_C', 'amount_cad_sum_D']
       customer_id  total_transaction_count  total_volume_cad  \
0  SYNID0100000167                       18          11562.53   
1  SYNID0100000431                       94          17939.29   
2  SYNID0100000485                       38          10204.58   
3  SYNID0100000539                      150          10409.15   
4  SYNID0100000932                      217         173363.91   

   avg_transaction_amount  median_transactio

In [172]:
customer_features.head()

,customer_id,total_transaction_count,total_volume_cad,avg_transaction_amount,median_transaction_amount,std_transaction_amount,min_transaction_amount,max_transaction_amount,first_transaction_date,last_transaction_date,...,card_count,cheque_count,eft_count,emt_count,western_union_count,wire_count,amount_cad_count_C,amount_cad_count_D,amount_cad_sum_C,amount_cad_sum_D
0,SYNID0100000167,18,11562.53,642.362778,276.66,816.759765,3.01,3236.46,2024-11-03 08:17:47,2025-01-31 16:42:13,...,0,1,17,0,0,0,10,8,9863.85,1698.68
1,SYNID0100000431,94,17939.29,190.843511,54.45,332.347397,0.00,1753.40,2024-11-01 18:04:19,2025-01-29 08:38:15,...,45,0,20,28,1,0,19,75,6475.51,11463.78
2,SYNID0100000485,38,10204.58,268.541579,106.81,441.820256,1.06,2264.88,2024-11-01 21:11:16,2025-01-31 16:42:13,...,19,0,19,0,0,0,11,27,6516.26,3688.32
3,SYNID0100000539,150,10409.15,69.394333,24.22,148.966923,0.00,1066.48,2024-11-02 12:20:05,2025-01-31 23:47:34,...,139,0,4,7,0,0,3,147,164.98,10244.17
4,SYNID0100000932,217,173363.91,798.912028,31.20,8549.518147,0.00,125397.63,2024-11-01 07:28:16,2025-01-31 17:01:45,...,189,2,25,1,0,0,23,194,145811.27,27552.64


In [173]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61403 entries, 0 to 61402
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   customer_id                61403 non-null  object        
 1   total_transaction_count    61403 non-null  int64         
 2   total_volume_cad           61403 non-null  float64       
 3   avg_transaction_amount     61403 non-null  float64       
 4   median_transaction_amount  61403 non-null  float64       
 5   std_transaction_amount     61403 non-null  float64       
 6   min_transaction_amount     61403 non-null  float64       
 7   max_transaction_amount     61403 non-null  float64       
 8   first_transaction_date     61403 non-null  datetime64[ns]
 9   last_transaction_date      61403 non-null  datetime64[ns]
 10  unique_active_days         61403 non-null  int64         
 11  account_activity_days      61403 non-null  int64         
 12  acti

In [174]:
final_features = customer_features.merge(
    kyc_individuals_df, 
    on='customer_id', 
    how='inner'  # Only keep customers who have both transaction AND KYC data
)

In [175]:
final_features.rename(columns={
    'amount_cad_count_C': 'credit_count',
    'amount_cad_sum_C': 'credit_volume',
    'amount_cad_count_D': 'debit_count',
    'amount_cad_sum_D': 'debit_volume'
}, inplace=True)

# Now your ratio calculations will work
final_features['total_spend_to_income_ratio'] = (
    final_features['debit_volume'] / (final_features['income'] + 1)
)

final_features['total_volume_to_income_ratio'] = (
    final_features['total_volume_cad'] / (final_features['income'] + 1)
)

In [176]:
# Calculate velocity variance
def calculate_velocity_variance(customer_transactions):
    customer_transactions = customer_transactions.sort_values('transaction_datetime')
    customer_transactions = customer_transactions.set_index('transaction_datetime')
    
    daily_counts = customer_transactions.resample('D').size()
    rolling_7day = daily_counts.rolling(window=7, min_periods=1).sum()
    velocity_variance = rolling_7day.var()
    
    return velocity_variance

# Apply to all customers
velocity_variance = all_transactions.groupby('customer_id').apply(
    calculate_velocity_variance,
    include_groups=False
).reset_index()
velocity_variance.columns = ['customer_id', 'velocity_variance_7day']

# Merge with final_features
final_features = final_features.merge(velocity_variance, on='customer_id', how='left')
final_features = final_features.assign(
    velocity_variance_7day=final_features['velocity_variance_7day'].fillna(0)
)

print("Velocity variance added!")
print(final_features[['customer_id', 'velocity_variance_7day']].head())

Velocity variance added!
       customer_id  velocity_variance_7day
0  SYNID0100000167                0.966916
1  SYNID0100000431                6.381523
2  SYNID0100000485                3.112637
3  SYNID0100000539               21.374115
4  SYNID0100000932               16.373626


In [177]:
final_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53093 entries, 0 to 53092
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_id                   53093 non-null  object        
 1   total_transaction_count       53093 non-null  int64         
 2   total_volume_cad              53093 non-null  float64       
 3   avg_transaction_amount        53093 non-null  float64       
 4   median_transaction_amount     53093 non-null  float64       
 5   std_transaction_amount        53093 non-null  float64       
 6   min_transaction_amount        53093 non-null  float64       
 7   max_transaction_amount        53093 non-null  float64       
 8   first_transaction_date        53093 non-null  datetime64[ns]
 9   last_transaction_date         53093 non-null  datetime64[ns]
 10  unique_active_days            53093 non-null  int64         
 11  account_activity_days       

In [178]:
final_features.head()

,customer_id,total_transaction_count,total_volume_cad,avg_transaction_amount,median_transaction_amount,std_transaction_amount,min_transaction_amount,max_transaction_amount,first_transaction_date,last_transaction_date,...,city,gender,marital_status,occupation_code,income,birth_date,onboard_date,total_spend_to_income_ratio,total_volume_to_income_ratio,velocity_variance_7day
0,SYNID0100000167,18,11562.53,642.362778,276.66,816.759765,3.01,3236.46,2024-11-03 08:17:47,2025-01-31 16:42:13,...,TORONTO,NaN,Married,10019,48886.0,1972-01-30,2011-09-20,0.034747,0.236515,0.966916
1,SYNID0100000431,94,17939.29,190.843511,54.45,332.347397,0.00,1753.40,2024-11-01 18:04:19,2025-01-29 08:38:15,...,other,FEMALE,Married,72310,NaN,1988-03-20,2018-06-04,NaN,NaN,6.381523
2,SYNID0100000485,38,10204.58,268.541579,106.81,441.820256,1.06,2264.88,2024-11-01 21:11:16,2025-01-31 16:42:13,...,BRAMPTON,FEMALE,Widowed,RETIRED,19998.0,1935-05-02,1997-07-12,0.184425,0.510255,3.112637
3,SYNID0100000539,150,10409.15,69.394333,24.22,148.966923,0.00,1066.48,2024-11-02 12:20:05,2025-01-31 23:47:34,...,other,MALE,Widowed,RETIRED,39417.0,1944-11-04,1985-05-14,0.259886,0.264071,21.374115
4,SYNID0100000932,217,173363.91,798.912028,31.20,8549.518147,0.00,125397.63,2024-11-01 07:28:16,2025-01-31 17:01:45,...,TORONTO,FEMALE,Married,RETIRED,34182.0,1963-09-14,2012-07-09,0.806033,5.071641,16.373626


In [180]:
final_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53093 entries, 0 to 53092
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_id                   53093 non-null  object        
 1   total_transaction_count       53093 non-null  int64         
 2   total_volume_cad              53093 non-null  float64       
 3   avg_transaction_amount        53093 non-null  float64       
 4   median_transaction_amount     53093 non-null  float64       
 5   std_transaction_amount        53093 non-null  float64       
 6   min_transaction_amount        53093 non-null  float64       
 7   max_transaction_amount        53093 non-null  float64       
 8   first_transaction_date        53093 non-null  datetime64[ns]
 9   last_transaction_date         53093 non-null  datetime64[ns]
 10  unique_active_days            53093 non-null  int64         
 11  account_activity_days       

In [181]:
print(final_features.head())

       customer_id  total_transaction_count  total_volume_cad  \
0  SYNID0100000167                       18          11562.53   
1  SYNID0100000431                       94          17939.29   
2  SYNID0100000485                       38          10204.58   
3  SYNID0100000539                      150          10409.15   
4  SYNID0100000932                      217         173363.91   

   avg_transaction_amount  median_transaction_amount  std_transaction_amount  \
0              642.362778                     276.66              816.759765   
1              190.843511                      54.45              332.347397   
2              268.541579                     106.81              441.820256   
3               69.394333                      24.22              148.966923   
4              798.912028                      31.20             8549.518147   

   min_transaction_amount  max_transaction_amount first_transaction_date  \
0                    3.01                 3236.46   

## PreProcessing and Fit the Model

In [183]:
final_features.head()

,customer_id,total_transaction_count,total_volume_cad,avg_transaction_amount,median_transaction_amount,std_transaction_amount,min_transaction_amount,max_transaction_amount,first_transaction_date,last_transaction_date,...,city,gender,marital_status,occupation_code,income,birth_date,onboard_date,total_spend_to_income_ratio,total_volume_to_income_ratio,velocity_variance_7day
0,SYNID0100000167,18,11562.53,642.362778,276.66,816.759765,3.01,3236.46,2024-11-03 08:17:47,2025-01-31 16:42:13,...,TORONTO,NaN,Married,10019,48886.0,1972-01-30,2011-09-20,0.034747,0.236515,0.966916
1,SYNID0100000431,94,17939.29,190.843511,54.45,332.347397,0.00,1753.40,2024-11-01 18:04:19,2025-01-29 08:38:15,...,other,FEMALE,Married,72310,NaN,1988-03-20,2018-06-04,NaN,NaN,6.381523
2,SYNID0100000485,38,10204.58,268.541579,106.81,441.820256,1.06,2264.88,2024-11-01 21:11:16,2025-01-31 16:42:13,...,BRAMPTON,FEMALE,Widowed,RETIRED,19998.0,1935-05-02,1997-07-12,0.184425,0.510255,3.112637
3,SYNID0100000539,150,10409.15,69.394333,24.22,148.966923,0.00,1066.48,2024-11-02 12:20:05,2025-01-31 23:47:34,...,other,MALE,Widowed,RETIRED,39417.0,1944-11-04,1985-05-14,0.259886,0.264071,21.374115
4,SYNID0100000932,217,173363.91,798.912028,31.20,8549.518147,0.00,125397.63,2024-11-01 07:28:16,2025-01-31 17:01:45,...,TORONTO,FEMALE,Married,RETIRED,34182.0,1963-09-14,2012-07-09,0.806033,5.071641,16.373626


In [186]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
df_processed = final_features.copy()

customer_ids = df_processed['customer_id'].copy()
df_processed = df_processed.drop('customer_id', axis=1)

datetime_cols = df_processed.select_dtypes(include=['datetime64']).columns.tolist()
for col in datetime_cols:
    df_processed[f'{col}_days_ago'] = (pd.Timestamp.now() - df_processed[col]).dt.days
    df_processed = df_processed.drop(col, axis=1)

if 'birth_date' in df_processed.columns:
    df_processed['birth_date'] = pd.to_datetime(df_processed['birth_date'], errors='coerce')
    df_processed['age'] = (pd.Timestamp.now() - df_processed['birth_date']).dt.days / 365.25
    df_processed = df_processed.drop('birth_date', axis=1)

if 'onboard_date' in df_processed.columns:
    df_processed['onboard_date'] = pd.to_datetime(df_processed['onboard_date'], errors='coerce')
    df_processed['customer_tenure_days'] = (pd.Timestamp.now() - df_processed['onboard_date']).dt.days
    df_processed = df_processed.drop('onboard_date', axis=1)

categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = df_processed[col].fillna('MISSING')
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le

numerical_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
for col in numerical_cols:
    if df_processed[col].isnull().any():
        df_processed[col].fillna(df_processed[col].median(), inplace=True)

print(f"Processed {df_processed.shape[1]} features for {df_processed.shape[0]} customers")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_processed)
feature_names = df_processed.columns.tolist()

print(f"Features scaled and ready for modeling")

iso_forest = IsolationForest(
    contamination=0.03,  # 3% of customers flagged as anomalies
    random_state=42,
    n_estimators=200,
    max_samples='auto',
    n_jobs=-1,
    verbose=0
)

# Fit and predict
predictions = iso_forest.fit_predict(X_scaled)
anomaly_scores_raw = iso_forest.score_samples(X_scaled)

print(f"Isolation Forest fitted with {iso_forest.n_estimators} trees")

# Convert anomaly scores to 0-100 risk scores (higher = riskier)
min_score = anomaly_scores_raw.min()
max_score = anomaly_scores_raw.max()
risk_scores = (1 - (anomaly_scores_raw - min_score) / (max_score - min_score)) * 100

# Create results dataframe
risk_results = pd.DataFrame({
    'customer_id': customer_ids,
    'anomaly_score': anomaly_scores_raw,
    'risk_score': risk_scores,
    'is_anomaly': (predictions == -1).astype(int),
    'risk_category': pd.cut(
        risk_scores, 
        bins=[0, 25, 50, 75, 100], 
        labels=['Low', 'Medium', 'High', 'Critical']
    )
})

# Sort by risk score
risk_results = risk_results.sort_values('risk_score', ascending=False).reset_index(drop=True)

print(f"Risk scores assigned to all {len(risk_results)} customers")
print(f"\nRisk Distribution:")
print(risk_results['risk_category'].value_counts().sort_index())
print(f"\nAnomalies flagged: {risk_results['is_anomaly'].sum()} ({risk_results['is_anomaly'].mean()*100:.2f}%)")

# Display top 10 riskiest customers
print(f"\nTop 10 Riskiest Customers:")
print(risk_results[['customer_id', 'risk_score', 'risk_category', 'is_anomaly']].head(10))


# Shap explanation
sample_size = min(500, len(X_scaled))
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

print(f"Computing SHAP values on sample of {sample_size} customers...")

# Create explainer (TreeExplainer works with Isolation Forest)
explainer = shap.Explainer(
    lambda x: -iso_forest.score_samples(x),  # Negative because lower score = more anomalous
    X_sample,
    feature_names=feature_names
)

# Calculate SHAP values
shap_values = explainer(X_sample)

print("SHAP values calculated")

print("\nGenerating SHAP Summary Plot (Feature Importance)...")
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values.values, X_sample, feature_names=feature_names, show=False)
plt.title("SHAP Feature Importance - Global View", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: shap_summary_plot.png")

# 2. Bar Plot - Average absolute SHAP values (feature importance)
print("\nGenerating SHAP Bar Plot (Top Features)...")
plt.figure(figsize=(12, 8))
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title("Top 20 Most Important Features (Mean |SHAP|)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: shap_bar_plot.png")

# 3. Calculate feature importance from SHAP
feature_importance_shap = pd.DataFrame({
    'feature': feature_names,
    'importance': np.abs(shap_values.values).mean(axis=0)
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (by SHAP):")
print(feature_importance_shap.head(20).to_string(index=False))

# Save feature importance
feature_importance_shap.to_csv('shap_feature_importance.csv', index=False)
print("\nSaved: shap_feature_importance.csv")

# Get top 5 riskiest customers from our sample
top_risky_indices = risk_results.iloc[sample_indices].nlargest(5, 'risk_score').index.tolist()
top_risky_sample_indices = [np.where(sample_indices == idx)[0][0] for idx in top_risky_indices if idx in sample_indices]

if len(top_risky_sample_indices) > 0:
    # Waterfall plot for top risky customer
    top_customer_idx = top_risky_sample_indices[0]
    top_customer_id = customer_ids.iloc[sample_indices[top_customer_idx]]
    top_risk_score = risk_results.loc[sample_indices[top_customer_idx], 'risk_score']
    
    print(f"\nExplaining Top Risk Customer: {top_customer_id} (Risk Score: {top_risk_score:.2f})")
    
    plt.figure(figsize=(12, 8))
    shap.plots.waterfall(shap_values[top_customer_idx], max_display=15, show=False)
    plt.title(f"SHAP Explanation - Customer {top_customer_id}\nRisk Score: {top_risk_score:.2f}", 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'shap_waterfall_top_customer.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: shap_waterfall_top_customer.png")
    
    print(f"\nGenerating SHAP Force Plot for {top_customer_id}...")
    try:
        # For PermutationExplainer, use base_values from shap_values object
        shap.plots.force(
            shap_values.base_values[top_customer_idx] if hasattr(shap_values, 'base_values') else 0, 
            shap_values.values[top_customer_idx], 
            X_sample[top_customer_idx],
            feature_names=feature_names,
            matplotlib=True,
            show=False
        )
        plt.title(f"SHAP Force Plot - Customer {top_customer_id}", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_force_plot_top_customer.png', dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved: shap_force_plot_top_customer.png")
    except Exception as e:
        print(f"Force plot not generated (this is optional): {str(e)}")
        print("   Continuing with other visualizations...")

def explain_customer_shap(customer_id):
    customer_idx = customer_ids[customer_ids == customer_id].index[0]
    
    # Check if in our sample
    if customer_idx not in sample_indices:
        print(f"Customer {customer_id} not in SHAP sample.")
        print("Computing SHAP for this customer...")
        
        # Compute SHAP for just this customer
        customer_data = X_scaled[customer_idx].reshape(1, -1)
        shap_single = explainer(customer_data)
        
        # Waterfall plot
        plt.figure(figsize=(12, 8))
        shap.plots.waterfall(shap_single[0], max_display=15, show=False)
        
        risk_info = risk_results[risk_results['customer_id'] == customer_id].iloc[0]
        plt.title(f"SHAP Explanation - Customer {customer_id}\n"
                 f"Risk Score: {risk_info['risk_score']:.2f} | Category: {risk_info['risk_category']}", 
                 fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_customer_{customer_id}.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Saved: shap_customer_{customer_id}.png")
        shap_vals = shap_single.values[0]
        feature_contributions = pd.DataFrame({
            'feature': feature_names,
            'shap_value': shap_vals,
            'abs_shap': np.abs(shap_vals)
        }).sort_values('abs_shap', ascending=False)
        
        print(f"\nTop 10 Contributing Features for {customer_id}:")
        print(feature_contributions.head(10)[['feature', 'shap_value']].to_string(index=False))
        
    else:
        sample_idx = np.where(sample_indices == customer_idx)[0][0]
        
        plt.figure(figsize=(12, 8))
        shap.plots.waterfall(shap_values[sample_idx], max_display=15, show=False)
        
        risk_info = risk_results[risk_results['customer_id'] == customer_id].iloc[0]
        plt.title(f"SHAP Explanation - Customer {customer_id}\n"
                 f"Risk Score: {risk_info['risk_score']:.2f} | Category: {risk_info['risk_category']}", 
                 fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_customer_{customer_id}.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Saved: shap_customer_{customer_id}.png")
        
        shap_vals = shap_values.values[sample_idx]
        feature_contributions = pd.DataFrame({
            'feature': feature_names,
            'shap_value': shap_vals,
            'abs_shap': np.abs(shap_vals)
        }).sort_values('abs_shap', ascending=False)
        
        print(f"\nTop 10 Contributing Features for {customer_id}:")
        print(feature_contributions.head(10)[['feature', 'shap_value']].to_string(index=False))
        
    return feature_contributions

# Save risk scores
risk_results.to_csv('aml_customer_risk_scores.csv', index=False)
print("Saved: aml_customer_risk_scores.csv")

high_risk = risk_results[risk_results['risk_score'] >= 75].copy()
high_risk.to_csv('high_risk_customers_for_review.csv', index=False)
print(f"Saved: high_risk_customers_for_review.csv ({len(high_risk)} customers)")

Processed 35 features for 53093 customers
Features scaled and ready for modeling
Isolation Forest fitted with 200 trees
Risk scores assigned to all 53093 customers

Risk Distribution:
risk_category
Low         43810
Medium       7770
High         1397
Critical      115
Name: count, dtype: int64

Anomalies flagged: 1593 (3.00%)

Top 10 Riskiest Customers:
       customer_id  risk_score risk_category  is_anomaly
0  SYNID0108297552  100.000000      Critical           1
1  SYNID0107153762   97.790744      Critical           1
2  SYNID0100068815   96.425613      Critical           1
3  SYNID0104884444   96.274637      Critical           1
4  SYNID0103491795   92.792081      Critical           1
5  SYNID0106406171   92.732817      Critical           1
6  SYNID0107224735   91.247263      Critical           1
7  SYNID0108931521   90.979036      Critical           1
8  SYNID0101485997   90.636495      Critical           1
9  SYNID0105285853   90.598828      Critical           1
Computing SHAP v

PermutationExplainer explainer: 501it [01:30,  4.90it/s]                         


SHAP values calculated

Generating SHAP Summary Plot (Feature Importance)...
Saved: shap_summary_plot.png

Generating SHAP Bar Plot (Top Features)...
Saved: shap_bar_plot.png

Top 20 Most Important Features (by SHAP):
                        feature  importance
                     card_count    0.004087
                occupation_code    0.003845
          account_activity_days    0.003704
first_transaction_date_days_ago    0.003560
                   cheque_count    0.003408
 last_transaction_date_days_ago    0.003045
                    debit_count    0.003017
       avg_transactions_per_day    0.003017
         velocity_variance_7day    0.002894
                         income    0.002783
                 marital_status    0.002782
                           city    0.002698
         min_transaction_amount    0.002531
                      emt_count    0.002530
         max_transaction_amount    0.002434
                  credit_volume    0.002416
                   debit_volume   

In [187]:
import os
print(os.path.getsize('aml_customer_risk_scores.csv'))

3253317
